# 3D Binary Membrane Mask Editor

This notebook is a simplified version of the previous workflow and is dedicated to **one binary 3D channel**.

### Purpose

Use a Magic-Wand-like workflow to edit a black/white volume stack and remove unwanted **outer membrane / shell structures** from a segmentation mask.

The notebook intentionally removes the previous lysosome/cell/PIMI analysis, second-channel processing, fluorescence quantification, blob detection, videos, and unrelated exports.

### Workflow

1. Load one binary 3D mask (`.mrc`, `.mrcs`, `.tif`, or `.tiff`).
2. Keep only the first/only channel.
3. Open the mask in Napari.
4. Move to the membrane you want to remove.
5. Place the cursor on the white structure and press **`W`** to select its connected 3D component.
6. The selected component is highlighted in red.
7. Press **`D`** to delete it from the working mask.
8. Press **`U`** to undo.
9. Save the cleaned mask as MRC and/or TIFF.

> **Important:** The original file is never modified. All edits are performed on an in-memory copy and saved to a new output file.


## Installation

Recommended environment: Anaconda / conda.

```bash
pip install numpy scipy scikit-image tifffile mrcfile napari magicgui
```

If Napari is installed with Qt dependencies, you may also need:

```bash
pip install "napari[all]"
```

For a full Anaconda installation workflow, see the main repository README.


In [ ]:
%pip install mrcfile

In [ ]:

# ============================================================
# 3D BINARY MEMBRANE MASK EDITOR
# ============================================================
#
# Supported input:
#   - MRC / MRCS
#   - TIFF / TIF
#
# The editor keeps ONE binary channel only.
#
# Napari keys:
#   W = Magic Wand: select the 3D connected component at cursor
#   D = Delete the selected component
#   U = Undo last deletion
#   S = Save cleaned mask
#   R = Reset working mask to original
#
# Optional:
#   A = Analyze current component as a possible shell
#
# ============================================================

import os
from pathlib import Path
from copy import deepcopy

import numpy as np
import tifffile
import napari

from scipy import ndimage as ndi
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects

try:
    import mrcfile
except ImportError as exc:
    raise ImportError(
        "The 'mrcfile' package is required for MRC input/output. "
        "Install it with: pip install mrcfile"
    ) from exc


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_COMPONENT_VOXELS = 10
CONNECTIVITY = 3          # 3x3x3 => 26-connectivity in 3D
BINARY_THRESHOLD = 0      # > this value is considered foreground


# ------------------------------------------------------------
# File I/O
# ------------------------------------------------------------

def load_volume(path):
    """Load one 3D volume and convert it to a binary mask."""
    path = str(path)
    ext = Path(path).suffix.lower()

    if ext in {".mrc", ".mrcs"}:
        with mrcfile.open(path, permissive=True) as mrc:
            data = np.array(mrc.data)
    elif ext in {".tif", ".tiff"}:
        data = tifffile.imread(path)
    else:
        raise ValueError(
            f"Unsupported file type: {ext}. "
            "Use .mrc, .mrcs, .tif or .tiff."
        )

    data = np.asarray(data)

    # Reduce singleton dimensions.
    data = np.squeeze(data)

    if data.ndim != 3:
        raise ValueError(
            f"Expected a 3D volume, but received shape {data.shape}."
        )

    # Keep ONLY ONE binary channel.
    # If a file contains an extra singleton/channel axis, squeeze() above
    # removes it when possible. Multi-channel non-singleton data is rejected
    # rather than silently selecting the wrong channel.
    mask = data > BINARY_THRESHOLD

    # Remove tiny isolated foreground objects.
    if MIN_COMPONENT_VOXELS > 1:
        mask = remove_small_objects(mask, min_size=MIN_COMPONENT_VOXELS)

    return mask.astype(np.uint8)


def save_volume(mask, path):
    """Save the cleaned binary mask as MRC or TIFF."""
    path = str(path)
    ext = Path(path).suffix.lower()

    out = (np.asarray(mask) > 0).astype(np.uint8)

    if ext in {".mrc", ".mrcs"}:
        with mrcfile.new(path, overwrite=True) as mrc:
            mrc.set_data(out)
    elif ext in {".tif", ".tiff"}:
        tifffile.imwrite(path, out, imagej=True)
    else:
        raise ValueError("Output must end in .mrc, .mrcs, .tif or .tiff.")

    print(f"Saved: {os.path.abspath(path)}")


# ------------------------------------------------------------
# Interactive editor
# ------------------------------------------------------------

class MembraneMaskEditor:
    def __init__(self, mask, source_path=None):
        self.original_mask = mask.copy().astype(bool)
        self.mask = mask.copy().astype(bool)
        self.source_path = source_path

        # Current selected component.
        self.selected_mask = np.zeros_like(self.mask, dtype=bool)

        # Undo stack contains full mask copies.
        self.undo_stack = []

        self.viewer = napari.Viewer(title="3D Binary Membrane Mask Editor")

        self.mask_layer = self.viewer.add_labels(
            self.mask.astype(np.uint8),
            name="WORKING MASK",
            opacity=0.75,
        )

        self.selection_layer = self.viewer.add_labels(
            np.zeros_like(self.mask, dtype=np.uint8),
            name="MAGIC WAND SELECTION",
            opacity=0.9,
        )

        # A simple original/reference layer. Hidden initially.
        self.original_layer = self.viewer.add_labels(
            self.original_mask.astype(np.uint8),
            name="ORIGINAL MASK",
            opacity=0.15,
            visible=False,
        )

        # Napari shortcuts.
        self.viewer.bind_key("W")(self.magic_wand)
        self.viewer.bind_key("D")(self.delete_selected)
        self.viewer.bind_key("U")(self.undo)
        self.viewer.bind_key("R")(self.reset)
        self.viewer.bind_key("S")(self.save)
        self.viewer.bind_key("A")(self.analyze_selected)

        print("===========================================")
        print("3D BINARY MEMBRANE MASK EDITOR")
        print("===========================================")
        print("W = select connected 3D component")
        print("D = delete selected component")
        print("U = undo")
        print("R = reset to original")
        print("S = save cleaned mask")
        print("A = analyze selected component")
        print("===========================================")

    def _cursor_index(self):
        """Convert Napari cursor position to integer z,y,x indices."""
        pos = np.asarray(self.viewer.cursor.position, dtype=float)

        if pos.size != 3:
            return None

        idx = np.rint(pos).astype(int)

        if np.any(idx < 0) or np.any(idx >= np.array(self.mask.shape)):
            return None

        return tuple(idx.tolist())

    def magic_wand(self, event=None):
        """Select the full 26-connected 3D component under the cursor."""
        idx = self._cursor_index()

        if idx is None:
            print("[Magic Wand] Cursor outside volume.")
            return

        if not self.mask[idx]:
            self.selected_mask[:] = False
            self.selection_layer.data = np.zeros_like(self.mask, dtype=np.uint8)
            print("[Magic Wand] Cursor is on background.")
            return

        # Connected component from the current working mask.
        labels = ndi.label(
            self.mask,
            structure=np.ones((3, 3, 3), dtype=np.uint8)
        )[0]

        component_id = labels[idx]

        if component_id == 0:
            print("[Magic Wand] No component found.")
            return

        self.selected_mask = labels == component_id
        self.selection_layer.data = self.selected_mask.astype(np.uint8)

        voxel_count = int(self.selected_mask.sum())
        zyx = np.argwhere(self.selected_mask)
        zmin, ymin, xmin = zyx.min(axis=0)
        zmax, ymax, xmax = zyx.max(axis=0)

        print(
            f"[Magic Wand] Selected component {component_id}: "
            f"{voxel_count:,} voxels | "
            f"Z {zmin}-{zmax}, Y {ymin}-{ymax}, X {xmin}-{xmax}"
        )

    def delete_selected(self, event=None):
        """Delete the selected component from the working mask."""
        n = int(self.selected_mask.sum())

        if n == 0:
            print("[Delete] Nothing selected.")
            return

        self.undo_stack.append(self.mask.copy())

        self.mask[self.selected_mask] = False
        self.mask_layer.data = self.mask.astype(np.uint8)

        self.selected_mask[:] = False
        self.selection_layer.data = np.zeros_like(self.mask, dtype=np.uint8)

        print(f"[Delete] Removed {n:,} voxels.")

    def undo(self, event=None):
        """Undo the last deletion."""
        if not self.undo_stack:
            print("[Undo] Nothing to undo.")
            return

        self.mask = self.undo_stack.pop()
        self.mask_layer.data = self.mask.astype(np.uint8)

        self.selected_mask[:] = False
        self.selection_layer.data = np.zeros_like(self.mask, dtype=np.uint8)

        print("[Undo] Restored previous mask.")

    def reset(self, event=None):
        """Reset the working mask to the original input."""
        self.undo_stack.append(self.mask.copy())
        self.mask = self.original_mask.copy()
        self.mask_layer.data = self.mask.astype(np.uint8)

        self.selected_mask[:] = False
        self.selection_layer.data = np.zeros_like(self.mask, dtype=np.uint8)

        print("[Reset] Working mask restored to original.")

    def analyze_selected(self, event=None):
        """
        Analyze the selected component for shell-like properties.

        This is a suggestion tool only; it does not delete anything.
        """
        n = int(self.selected_mask.sum())

        if n == 0:
            print("[Analyze] Nothing selected.")
            return

        # Thickness proxy from Euclidean distance to background.
        dist = ndi.distance_transform_edt(self.selected_mask)
        max_half_thickness = float(dist.max())
        approx_diameter_vox = 2.0 * max_half_thickness

        # Fill the holes inside the component's bounding box.
        zyx = np.argwhere(self.selected_mask)
        z0, y0, x0 = zyx.min(axis=0)
        z1, y1, x1 = zyx.max(axis=0)

        crop = self.selected_mask[z0:z1+1, y0:y1+1, x0:x1+1]
        filled = ndi.binary_fill_holes(crop)

        filled_volume = int(filled.sum())
        shell_volume = int(crop.sum())

        fill_ratio = filled_volume / max(shell_volume, 1)

        print("===========================================")
        print("Selected component analysis")
        print("===========================================")
        print(f"Voxel count              : {n:,}")
        print(f"Approx. max thickness   : {approx_diameter_vox:.2f} voxels")
        print(f"Filled volume            : {filled_volume:,}")
        print(f"Shell volume             : {shell_volume:,}")
        print(f"Filled / shell volume   : {fill_ratio:.3f}")
        print("===========================================")
        print(
            "Higher filled/shell ratio can indicate a thin shell "
            "surrounding a larger enclosed volume."
        )

    def save(self, event=None):
        """Save a cleaned copy next to the original file."""
        if self.source_path:
            src = Path(self.source_path)
            out = src.with_name(src.stem + "_cleaned" + src.suffix)
        else:
            out = Path("mask_cleaned.mrc")

        save_volume(self.mask, out)

    def run(self):
        napari.run()


# ------------------------------------------------------------
# Select input
# ------------------------------------------------------------

from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

input_path = filedialog.askopenfilename(
    title="Select ONE binary 3D mask",
    filetypes=[
        ("MRC files", "*.mrc *.mrcs"),
        ("TIFF files", "*.tif *.tiff"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if not input_path:
    raise RuntimeError("No input file selected.")

mask = load_volume(input_path)

print("Loaded:", input_path)
print("Shape :", mask.shape)
print("Dtype :", mask.dtype)
print("Foreground voxels:", int(mask.sum()))

editor = MembraneMaskEditor(mask, source_path=input_path)
editor.run()


## How to use the editor

### Select an outer membrane / shell

1. Open the notebook and run the main cell.
2. Load your black/white 3D segmentation.
3. In Napari, navigate to the Z slice containing the membrane.
4. Place the cursor on the white membrane you want to remove.
5. Press **`W`**.
6. The complete 3D connected component will appear in the **MAGIC WAND SELECTION** layer.

### Remove it

Press:

```text
D
```

The selected component will be removed from the **WORKING MASK**.

### Undo

Press:

```text
U
```

### Restore the original mask

Press:

```text
R
```

### Analyze a selected structure

Press:

```text
A
```

This reports simple shell-like measurements such as voxel volume, approximate thickness, and filled-to-shell volume ratio. It **does not delete anything automatically**.

### Save

Press:

```text
S
```

The cleaned mask is saved next to the original file with:

```text
_original_name_cleaned.mrc
```

or the equivalent TIFF filename.

## Notes

- The tool operates on **one binary 3D channel only**.
- The original input is never overwritten.
- Selection uses **26-connectivity in 3D**, so diagonally touching voxels are considered connected.
- The first version intentionally uses a **semi-automatic** workflow: the user selects the structure, while Python performs the 3D component selection and deletion.
- Automatic classification of outer vs. inner membranes can be added later using shell thickness, enclosed-volume, and containment features.
